# Iniciando o Spark

In [1]:
## Bloco de codigo para instalar versao especifica dos pacotes
!pip install pyspark

In [2]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Trusted_base_recarga") \
    .getOrCreate()

# Importando bibliotecas

In [4]:
import os
import pytz
import datetime
from datetime import datetime
#from pyspark.sql.types import *
#from pyspark.sql.functions import count, avg
#import sys
#import numpy as np
#from datetime import datetime
#from pyspark.sql import SQLContext
#from datetime import timedelta
#from datetime import date
#from dateutil.relativedelta import relativedelta
#from pyspark.sql.functions import udf, lpad, translate

# Funções auxiliares e variáveis

In [6]:
# Função de log
def log():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S') + " >>>"

# Timestamp de processamento (com hora/minuto/segundo)
agora = datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc = agora.strftime("%Y%m%d%H%M%S")

# Data da execução (AAAAmmdd)
PROCESS_DATE = datetime.now().strftime("%Y%m%d")

# Período de referência (AAAAmm)
REF_PERIOD = datetime.now().strftime("%Y%m")

#Alterar o path_padrao caso seus arquivos não estejam nesse mesmo caminho
path_padrao = "/content/gdrive/Othercomputers/Meu laptop"

# Buckets e nomes de saída
bucket_base = "base_recarga"
bucket_raw = f"{path_padrao}/Database_raw/bases_recarga"
bucket_raw_dim = f"{path_padrao}/Database_raw/"
bucket_trusted = f"{path_padrao}/Database_trusted/bases_recarga"
bucket_control = f"{path_padrao}/Database_control/bases_recarga"
output_trusted = f"trusted_{bucket_base}"

# Prints para conferência
print("PROCESS_DATE:", PROCESS_DATE)
print("REF_PERIOD:", REF_PERIOD)
print("dthproc:", dthproc)
print("bucket_trusted:", bucket_trusted)
print("bucket_raw:", bucket_raw)
print("bucket_control:", bucket_control)


PROCESS_DATE: 20260221
REF_PERIOD: 202602
dthproc: 20260221183326
bucket_trusted: /content/gdrive/Othercomputers/Meu laptop/Database_trusted/bases_recarga
bucket_raw: /content/gdrive/Othercomputers/Meu laptop/Database_raw/bases_recarga
bucket_control: /content/gdrive/Othercomputers/Meu laptop/Database_control/bases_recarga


#  Leitura da camada Raw

In [7]:
path_raw = os.path.join(bucket_raw, "BI_FP_ASS_RECARGA_CMV_NOVA")

parquet_files = [path_raw for f in os.listdir(path_raw) if f.endswith('.parquet')]
df_raw_recarga= spark.read.parquet(*parquet_files, header=True, inferSchema=True)
df_raw_recarga.createOrReplaceTempView("raw_base_recarga")

print(log(), "Registros na Raw:", df_raw_recarga.count())
df_raw_recarga.show(5, truncate=False)

2026-02-21 21:33:36 >>> Registros na Raw: 1002136510
+-----------+----------+--------------------+--------------------+--------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+
|NUM_CPF    |DW_NUM_NTC|DAT_INSERCAO_CREDITO|HOR_INSERCAO_CREDITO|DW_NUM_CLIENTE|COD_TECNOLOGIA_DW|COD_CANAL_AQUISICAO|COD_TIPO_CREDITO|COD_PROMOCAO|VAL_CREDITO_INSERIDO|VAL_BONUS|VAL_REAL|COD_PLATAFORMA_ATU|COD_STATUS_PLATAFORMA|IND_METODO_PAGAMENTO|DW_PLANO_TARIFACAO|DW_TIPO_RECARGA|DW_TIPO_INSERCAO|DW_FORMA_PAGAMENTO|DW_INSTITUICAO|COD_GRUPO_CARTAO|DSC_GRUPO_CARTAO_WPP|FLAG_SOS|VALOR_SOS|
+-----------+----------+--------------------+--------------------+--------------+-----------------+-------------------+----------------+------------+--------

### Montando as dimensões para enriquecer a base

In [ ]:
PATH_DIMENSOES_RECARGA = {
    "CANAL_AQUISICAO": f"{bucket_raw_dim}/BI_DIM_CANAL_AQUISICAO_CREDITO.csv",
    "FORMA_PAGAMENTO": f"{bucket_raw_dim}/BI_DIM_FORMA_PAGAMENTO.csv",
    "INSTITUICAO": f"{bucket_raw_dim}/BI_DIM_INSTITUICAO.csv",
    "PLATAFORMA": f"{bucket_raw_dim}/BI_DIM_PLATAFORMA.csv",
    "PROMOCAO": f"{bucket_raw_dim}/BI_DIM_PROMOCAO_CREDITO.csv",
    "STATUS_PLATAFORMA": f"{bucket_raw_dim}/BI_DIM_STATUS_PLATAFORMA.csv",
    "TECNOLOGIA": f"{bucket_raw_dim}/BI_DIM_TECNOLOGIA.csv",
    "TIPO_CREDITO": f"{bucket_raw_dim}/BI_DIM_TIPO_CREDITO.csv",
    "TIPO_INSERCAO": f"{bucket_raw_dim}/BI_DIM_TIPO_INSERCAO.csv",
    "TIPO_RECARGA": f"{bucket_raw_dim}/BI_DIM_TIPO_RECARGA.csv",
}

DIMENSOES_RECARGA = {
    "CANAL_AQUISICAO_CREDITO": "BI_DIM_CANAL_AQUISICAO_CREDITO.csv",
    "FORMA_PAGAMENTO": "BI_DIM_FORMA_PAGAMENTO.csv",
    "INSTITUICAO": "BI_DIM_INSTITUICAO.csv",
    "PLANO_PRECO": "BI_DIM_PLANO_PRECO.csv",
    "PLATAFORMA": "BI_DIM_PLATAFORMA.csv",
    "PROMOCAO_CREDITO": "BI_DIM_PROMOCAO_CREDITO.csv",
    "STATUS_PLATAFORMA": "BI_DIM_STATUS_PLATAFORMA.csv",
    "TECNOLOGIA": "BI_DIM_TECNOLOGIA.csv",
    "TIPO_CREDITO": "BI_DIM_TIPO_CREDITO.csv",
    "TIPO_INSERCAO": "BI_DIM_TIPO_INSERCAO.csv",
    "TIPO_RECARGA": "BI_DIM_TIPO_RECARGA.csv",
}


dfs_dim_recarga = {}

for nome_dim, arquivo in DIMENSOES_RECARGA.items():
    path = f"{bucket_raw}/{arquivo}"

    dfs_dim_recarga[nome_dim] = (
        spark.read
        .option("header", True)
        .option("sep", ",")
        .option("inferSchema", True)
        .csv(path)
    )

df_CANAL_AQUISICAO_CREDITO = dfs_dim_recarga["CANAL_AQUISICAO_CREDITO"]
df_FORMA_PAGAMENTO        = dfs_dim_recarga["FORMA_PAGAMENTO"]
df_INSTITUICAO            = dfs_dim_recarga["INSTITUICAO"]
df_PLANO_PRECO            = dfs_dim_recarga["PLANO_PRECO"]
df_PLATAFORMA             = dfs_dim_recarga["PLATAFORMA"]
df_PROMOCAO_CREDITO       = dfs_dim_recarga["PROMOCAO_CREDITO"]
df_STATUS_PLATAFORMA      = dfs_dim_recarga["STATUS_PLATAFORMA"]
df_TECNOLOGIA              = dfs_dim_recarga["TECNOLOGIA"]
df_TIPO_CREDITO           = dfs_dim_recarga["TIPO_CREDITO"]
df_TIPO_INSERCAO          = dfs_dim_recarga["TIPO_INSERCAO"]
df_TIPO_RECARGA           = dfs_dim_recarga["TIPO_RECARGA"]

df_TIPO_RECARGA.show(5)

+---------------+--------------------+----------------+------------------+
|DW_TIPO_RECARGA|    DSC_TIPO_RECARGA|DAT_EXPIRACAO_DW|    DAT_CRIACAO_DW|
+---------------+--------------------+----------------+------------------+
|             -3|       Não Informado|            NULL|19OCT2011:14:39:56|
|             -2|     Não Determinado|            NULL|19OCT2011:14:39:56|
|             -1|       Não se Aplica|            NULL|19OCT2011:14:39:56|
|              1|Franquia do Claro...|            NULL|19OCT2011:14:39:56|
|              2|Adicional do Clar...|            NULL|19OCT2011:14:39:56|
+---------------+--------------------+----------------+------------------+



In [ ]:
# optamos por não inserir as dimensões nessa camada,
# para manter a granularidade e flexibilidade da base,
# e evitar possíveis problemas de atualização das dimensões.
# As dimensões serão integradas em análises futuras na construção dos books
# garantindo que a camada Trusted permaneça o mais fiel possível à fonte original.

# Processamento tipagem para camada Trusted

In [22]:
df_base_recarga = spark.sql(f"""
    SELECT
        '{dthproc}' AS ts_proc,
        '{dthproc}' AS ts_proc_partition,
        CAST(NUM_CPF AS STRING) AS NUM_CPF,
        CAST(DW_NUM_NTC AS STRING) AS DW_NUM_NTC,
        CAST(DW_NUM_CLIENTE AS STRING) AS DW_NUM_CLIENTE,
        CAST(DAT_INSERCAO_CREDITO AS STRING) AS DAT_INSERCAO_CREDITO,
        CAST(HOR_INSERCAO_CREDITO AS STRING) AS HOR_INSERCAO_CREDITO,
        CAST(COD_TECNOLOGIA_DW AS STRING) AS COD_TECNOLOGIA_DW,
        CAST(COD_CANAL_AQUISICAO AS INT) AS COD_CANAL_AQUISICAO,
        CAST(COD_TIPO_CREDITO AS STRING) AS COD_TIPO_CREDITO,
        CAST(COD_PROMOCAO AS INT) AS COD_PROMOCAO,
        CAST(VAL_CREDITO_INSERIDO AS FLOAT) AS VAL_CREDITO_INSERIDO,
        CAST(VAL_BONUS AS FLOAT) AS VAL_BONUS,
        CAST(VAL_REAL AS FLOAT) AS VAL_REAL,
        CAST(COD_PLATAFORMA_ATU AS STRING) AS COD_PLATAFORMA_ATU,
        CAST(COD_STATUS_PLATAFORMA AS STRING) AS COD_STATUS_PLATAFORMA,
        CAST(IND_METODO_PAGAMENTO AS STRING) AS IND_METODO_PAGAMENTO,
        CAST(DW_PLANO_TARIFACAO AS INT) AS DW_PLANO_TARIFACAO,
        CAST(DW_TIPO_RECARGA AS INT) AS DW_TIPO_RECARGA,
        CAST(DW_TIPO_INSERCAO AS INT) AS DW_TIPO_INSERCAO,
        CAST(DW_FORMA_PAGAMENTO AS INT) AS DW_FORMA_PAGAMENTO,
        CAST(DW_INSTITUICAO AS INT) AS DW_INSTITUICAO,
        CAST(COD_GRUPO_CARTAO AS STRING) AS COD_GRUPO_CARTAO,
        CAST(DSC_GRUPO_CARTAO_WPP AS STRING) AS DSC_GRUPO_CARTAO_WPP,
        CAST(FLAG_SOS AS INT) AS FLAG_SOS,
        CAST(VALOR_SOS AS INT) AS VALOR_SOS
    FROM raw_base_recarga
""")

df_base_recarga.createOrReplaceTempView("df_base_recarga")
#df_base_recarga.cache()

total_recarga = df_base_recarga.count()
print(log(), "Registros Base:",total_recarga)
#df_base_recarga.printSchema()
df_base_recarga.show(5, truncate=False)

2026-02-21 22:10:27 >>> Registros Base: 1002136510
+--------------+-----------------+-----------+----------+--------------+--------------------+--------------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+
|ts_proc       |ts_proc_partition|NUM_CPF    |DW_NUM_NTC|DW_NUM_CLIENTE|DAT_INSERCAO_CREDITO|HOR_INSERCAO_CREDITO|COD_TECNOLOGIA_DW|COD_CANAL_AQUISICAO|COD_TIPO_CREDITO|COD_PROMOCAO|VAL_CREDITO_INSERIDO|VAL_BONUS|VAL_REAL|COD_PLATAFORMA_ATU|COD_STATUS_PLATAFORMA|IND_METODO_PAGAMENTO|DW_PLANO_TARIFACAO|DW_TIPO_RECARGA|DW_TIPO_INSERCAO|DW_FORMA_PAGAMENTO|DW_INSTITUICAO|COD_GRUPO_CARTAO|DSC_GRUPO_CARTAO_WPP|FLAG_SOS|VALOR_SOS|
+--------------+-----------------+-----------+----------+--------------+--------------------+

#Inserindo os dados das dimensões

In [25]:
#DIM CANAL_AQUISICAO_CREDITO
df_CANAL_AQUISICAO_CREDITO.createOrReplaceTempView("CANAL_AQUISICAO_CREDITO")
df_base_recarga_canal = spark.sql("""
    SELECT
        f.*,
        c.COD_CANAL_AQUISICAO AS CAN_COD_CANAL_AQUISICAO,
        c.DSC_CANAL_AQUISICAO AS CAN_DSC_CANAL_AQUISICAO,
        c.COD_SISTEMA_DW AS CAN_COD_SISTEMA_DW,
        c.DAT_ATUALIZACAO_DW AS CAN_DAT_ATUALIZACAO_DW,
        c.DAT_CRIACAO_DW AS CAN_DAT_CRIACAO_DW,
        c.COD_CANAL_AQUISICAO_BI AS CAN_COD_CANAL_AQUISICAO_BI,
        c.DSC_CANAL_AQUISICAO_BI AS CAN_DSC_CANAL_AQUISICAO_BI,
        c.COD_AGENTE_CREDITO AS CAN_COD_AGENTE_CREDITO,
        c.DAT_EXPIRACAO_DW AS CAN_DAT_EXPIRACAO_DW,
        c.COD_TIPO_CREDITO AS CAN_COD_TIPO_CREDITO,
        c.COD_TIPO_INSTITUICAO AS CAN_COD_TIPO_INSTITUICAO,
        c.DSC_TIPO_INSTITUICAO AS CAN_DSC_TIPO_INSTITUICAO
    FROM df_base_recarga f
    LEFT JOIN CANAL_AQUISICAO_CREDITO c
        ON f.COD_CANAL_AQUISICAO = c.COD_CANAL_AQUISICAO
""")

df_base_recarga_canal.createOrReplaceTempView("df_base_recarga_canal")
total_linhas = df_base_recarga_canal.count()

if (total_linhas == total_recarga):  #df_base_recarga_canal.count() == df_base_recarga.count()
    print(log(), "Join valido Registros Base:", total_linhas)
    df_base_recarga_canal.show(5)

2026-02-21 22:13:33 >>> Join valido Registros Base: 1002136510
+--------------+-----------------+-----------+----------+--------------+--------------------+--------------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+-----------------------+-----------------------+------------------+----------------------+------------------+--------------------------+--------------------------+----------------------+--------------------+--------------------+------------------------+------------------------+
|       ts_proc|ts_proc_partition|    NUM_CPF|DW_NUM_NTC|DW_NUM_CLIENTE|DAT_INSERCAO_CREDITO|HOR_INSERCAO_CREDITO|COD_TECNOLOGIA_DW|COD_CANAL_AQUISICAO|COD_TIPO_CREDITO|COD_PROMOCAO|VAL_CREDITO_INSERIDO|VAL_BONUS|VAL_REAL|COD_PLATA

In [26]:
#DIM FORMA_PAGAMENTO
df_FORMA_PAGAMENTO.createOrReplaceTempView("FORMA_PAGAMENTO")
df_base_recarga_forma = spark.sql("""
    SELECT
        f.*,
        fp.COD_FORMA_PAGAMENTO AS FP_COD_FORMA_PAGAMENTO,
        fp.DSC_FORMA_PAGAMENTO AS FP_DSC_FORMA_PAGAMENTO,
        fp.DAT_CRIACAO_DW AS FP_DAT_CRIACAO_DW,
        fp.DAT_EXPIRACAO_DW AS FP_DAT_EXPIRACAO_DW
    FROM df_base_recarga_canal f
    LEFT JOIN FORMA_PAGAMENTO fp
        ON f.DW_FORMA_PAGAMENTO = fp.DW_FORMA_PAGAMENTO
""")

df_base_recarga_forma.createOrReplaceTempView("df_base_recarga_forma")
total_linhas= df_base_recarga_forma.count()

if (total_linhas == total_recarga):
    print(log(), "Join valido Registros Base:", total_linhas)
    df_base_recarga_forma.show(5)

2026-02-21 22:16:29 >>> Join valido Registros Base: 1002136510
+--------------+-----------------+-----------+----------+--------------+--------------------+--------------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+-----------------------+-----------------------+------------------+----------------------+------------------+--------------------------+--------------------------+----------------------+--------------------+--------------------+------------------------+------------------------+----------------------+----------------------+------------------+-------------------+
|       ts_proc|ts_proc_partition|    NUM_CPF|DW_NUM_NTC|DW_NUM_CLIENTE|DAT_INSERCAO_CREDITO|HOR_INSERCAO_CREDITO|COD_TECNOLOGIA_DW|COD_CANAL_AQUI

In [29]:
#DIM INSTITUICAO
df_INSTITUICAO.createOrReplaceTempView("INSTITUICAO")
df_base_recarga_instituicao = spark.sql("""
    SELECT
        f.*,
        i.COD_INSTITUICAO AS INS_COD_INSTITUICAO,
        i.DSC_INSTITUICAO AS INS_DSC_INSTITUICAO,
        i.COD_TIPO_INSTITUICAO AS INS_COD_TIPO_INSTITUICAO,
        i.DSC_TIPO_INSTITUICAO AS INS_DSC_TIPO_INSTITUICAO,
        i.COD_SISTEMA_DW AS INS_COD_SISTEMA_DW,
        i.DAT_EXPIRACAO_DW AS INS_DAT_EXPIRACAO_DW,
        i.DAT_CRIACAO_DW AS INS_DAT_CRIACAO_DW,
        i.COD_AGENTE AS INS_COD_AGENTE
    FROM df_base_recarga_forma f
    LEFT JOIN INSTITUICAO i
        ON f.DW_INSTITUICAO = i.DW_INSTITUICAO
""")

df_base_recarga_instituicao.createOrReplaceTempView("df_base_recarga_instituicao")
total_linhas = df_base_recarga_instituicao.count()

if (total_linhas == total_recarga):
    print(log(), "Join valido Registros Base:", total_linhas)
    df_base_recarga_instituicao.show(5)

2026-02-21 22:30:18 >>> Join valido Registros Base: 1002136510
+--------------+-----------------+-----------+----------+--------------+--------------------+--------------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+-----------------------+-----------------------+------------------+----------------------+------------------+--------------------------+--------------------------+----------------------+--------------------+--------------------+------------------------+------------------------+----------------------+----------------------+------------------+-------------------+-------------------+-------------------+------------------------+------------------------+------------------+--------------------+-----------------

In [30]:
#DIM PLANO_PRECO
df_PLANO_PRECO.createOrReplaceTempView("PLANO_PRECO")
df_base_recarga_plano = spark.sql("""
    SELECT
        f.*,
        pp.DW_PLANO AS PP_DW_PLANO,
        pp.COD_PLANO_PRECO AS PP_COD_PLANO_PRECO,
        pp.DSC_PLANO_PRECO AS PP_DSC_PLANO_PRECO,
        pp.COD_TIPO_CLIENTE AS PP_COD_TIPO_CLIENTE,
        pp.COD_SUB_TIPO_CLIENTE AS PP_COD_SUB_TIPO_CLIENTE,
        pp.DW_TIPO_CLIENTE AS PP_DW_TIPO_CLIENTE,
        pp.DAT_EFETIVACAO AS PP_DAT_EFETIVACAO,
        pp.DAT_EXPIRACAO AS PP_DAT_EXPIRACAO,
        pp.DSC_PLANO_PRECO_BI AS PP_DSC_PLANO_PRECO_BI,
        pp.DSC_GRUPO_PLANO_BI AS PP_DSC_GRUPO_PLANO_BI,
        pp.DSC_TIPO_PLANO_BI AS PP_DSC_TIPO_PLANO_BI,
        pp.IND_AMDOCS_PLAT_PRE AS PP_IND_AMDOCS_PLAT_PRE,
        pp.COD_TRATAMENTO_ESPECIAL AS PP_COD_TRATAMENTO_ESPECIAL,
        pp.COD_SISTEMA_DW AS PP_COD_SISTEMA_DW,
        pp.COD_TECNOLOGIA_DW AS PP_COD_TECNOLOGIA_DW,
        pp.DAT_EXPIRACAO_DW AS PP_DAT_EXPIRACAO_DW,
        pp.DAT_ATUALIZACAO_DW AS PP_DAT_ATUALIZACAO_DW,
        pp.DAT_CRIACAO_DW AS PP_DAT_CRIACAO_DW,
        pp.NUM_FRANQUIA_MINUTOS_BI AS PP_NUM_FRANQUIA_MINUTOS_BI,
        pp.NUM_FRANQUIA_REAIS_BI AS PP_NUM_FRANQUIA_REAIS_BI,
        pp.NUM_FRANQUIA_EVENTOS_BI AS PP_NUM_FRANQUIA_EVENTOS_BI,
        pp.NUM_FRANQUIA_VOLUME_BI AS PP_NUM_FRANQUIA_VOLUME_BI,
        pp.COD_PLANO_COMPONENTE AS PP_COD_PLANO_COMPONENTE,
        pp.DSC_PLANO_PRECO_UNICO_BI AS PP_DSC_PLANO_PRECO_UNICO_BI,
        pp.DSC_MODALIDADE_PLANO AS PP_DSC_MODALIDADE_PLANO
    FROM df_base_recarga_instituicao f
    LEFT JOIN PLANO_PRECO pp
        ON CAST(f.DW_PLANO_TARIFACAO AS STRING) = CAST(pp.COD_PLANO_PRECO AS STRING)
""")

df_base_recarga_plano.createOrReplaceTempView("df_base_recarga_plano")
total_linhas = df_base_recarga_plano.count()

if (total_linhas == total_recarga):
    print(log(), "Join valido Registros Base:", total_linhas)
    df_base_recarga_plano.show(5)

2026-02-21 22:37:24 >>> Join valido Registros Base: 1002136510
+--------------+-----------------+-----------+----------+--------------+--------------------+--------------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+-----------------------+-----------------------+------------------+----------------------+------------------+--------------------------+--------------------------+----------------------+--------------------+--------------------+------------------------+------------------------+----------------------+----------------------+------------------+-------------------+-------------------+-------------------+------------------------+------------------------+------------------+--------------------+-----------------

In [31]:
#DIM STATUS_PLATAFORMA
df_STATUS_PLATAFORMA.createOrReplaceTempView("STATUS_PLATAFORMA")
df_base_plataforma = spark.sql("""
    SELECT
        f.*,
        sp.DSC_STATUS_PLATAFORMA AS SP_DSC_STATUS_PLATAFORMA,
        sp.IND_ATIVO AS SP_IND_ATIVO,
        sp.DAT_ATUALIZACAO_DW AS SP_DAT_ATUALIZACAO_DW,
        sp.DAT_CRIACAO_DW AS SP_DAT_CRIACAO_DW,
        sp.COD_STATUS_PLAT_GRP AS SP_COD_STATUS_PLAT_GRP,
        sp.IND_STS_PLAT_GRP_ATIVO AS SP_IND_STS_PLAT_GRP_ATIVO
    FROM df_base_recarga_plano f
    LEFT JOIN STATUS_PLATAFORMA sp
        ON f.COD_STATUS_PLATAFORMA = sp.COD_STATUS_PLATAFORMA
""")

df_base_plataforma.createOrReplaceTempView("df_base_plataforma")
total_linhas = df_base_plataforma.count()

if (total_linhas == total_recarga):
    print(log(), "Join valido Registros Base:", total_linhas)
    df_base_plataforma.show(5)

2026-02-21 22:49:07 >>> Join valido Registros Base: 1002136510
+--------------+-----------------+-----------+----------+--------------+--------------------+--------------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+-----------------------+-----------------------+------------------+----------------------+------------------+--------------------------+--------------------------+----------------------+--------------------+--------------------+------------------------+------------------------+----------------------+----------------------+------------------+-------------------+-------------------+-------------------+------------------------+------------------------+------------------+--------------------+-----------------

In [33]:
#DIM PROMOCAO
df_PROMOCAO_CREDITO.createOrReplaceTempView("PROMOCAO_CREDITO")
df_base_recarga_promocao = spark.sql("""
    SELECT
        f.*,
        pr.DSC_PROMOCAO AS PR_DSC_PROMOCAO,
        pr.DAT_EXPIRACAO_DW AS PR_DAT_EXPIRACAO_DW,
        pr.DAT_ATUALIZACAO_DW AS PR_DAT_ATUALIZACAO_DW,
        pr.DAT_CRIACAO_DW AS PR_DAT_CRIACAO_DW,
        pr.COD_PROM_GRUPO_CARTAO AS PR_COD_PROM_GRUPO_CARTAO,
        pr.DSC_NOME_PROMOCAO AS PR_DSC_NOME_PROMOCAO,
        pr.COD_TIPO_PROMOCAO AS PR_COD_TIPO_PROMOCAO,
        pr.DAT_INICIO_VIGENCIA AS PR_DAT_INICIO_VIGENCIA,
        pr.DAT_FIM_VIGENCIA AS PR_DAT_FIM_VIGENCIA,
        pr.VAL_PROMOCAO AS PR_VAL_PROMOCAO,
        pr.NUM_CONTA_DEDICADA AS PR_NUM_CONTA_DEDICADA
    FROM df_base_plataforma f
    LEFT JOIN PROMOCAO_CREDITO pr
        ON f.COD_PROMOCAO = pr.COD_PROMOCAO
""")

df_base_recarga_promocao.createOrReplaceTempView("df_base_recarga_promocao")
total_linhas = df_base_recarga_promocao.count()

if (total_linhas == total_recarga):
    print(log(), "Join valido Registros Base:", total_linhas)
    df_base_recarga_promocao.show(5)

2026-02-21 23:05:48 >>> Join valido Registros Base: 1002136510
+--------------+-----------------+-----------+----------+--------------+--------------------+--------------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+-----------------------+-----------------------+------------------+----------------------+------------------+--------------------------+--------------------------+----------------------+--------------------+--------------------+------------------------+------------------------+----------------------+----------------------+------------------+-------------------+-------------------+-------------------+------------------------+------------------------+------------------+--------------------+-----------------

In [34]:
#DIM TECNOLOGIA
df_TECNOLOGIA.createOrReplaceTempView("TECNOLOGIA")
df_base_recarga_tecnologia = spark.sql("""
    SELECT
        f.*,
        t.DSC_TECNOLOGIA AS TEC_DSC_TECNOLOGIA,
        t.DAT_ATUALIZACAO_DW AS TEC_DAT_ATUALIZACAO_DW,
        t.DAT_CRIACAO_DW AS TEC_DAT_CRIACAO_DW,
        t.COD_TECNOLOGIA_SVA AS TEC_COD_TECNOLOGIA_SVA
    FROM df_base_recarga_promocao f
    LEFT JOIN TECNOLOGIA t
        ON CAST(f.COD_TECNOLOGIA_DW AS STRING) = CAST(t.COD_TECNOLOGIA_DW AS STRING)
""")

df_base_recarga_tecnologia.createOrReplaceTempView("df_base_recarga_tecnologia")
total_linhas = df_base_recarga_tecnologia.count()

if (total_linhas == total_recarga):
    print(log(), "Join valido Registros Base:", total_linhas)
    df_base_recarga_tecnologia.show(5)

2026-02-21 23:21:23 >>> Join valido Registros Base: 1002136510
+--------------+-----------------+-----------+----------+--------------+--------------------+--------------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+-----------------------+-----------------------+------------------+----------------------+------------------+--------------------------+--------------------------+----------------------+--------------------+--------------------+------------------------+------------------------+----------------------+----------------------+------------------+-------------------+-------------------+-------------------+------------------------+------------------------+------------------+--------------------+-----------------

In [36]:
#DIM TIPO_CREDITO
df_TIPO_CREDITO.createOrReplaceTempView("TIPO_CREDITO")
# Perform the join using the renamed fact column
df_base_recarga_tipocredito = spark.sql("""
    SELECT
        f.*,
        tc.COD_TIPO_CREDITO AS TPC_COD_TIPO_CREDITO,
        tc.DSC_TIPO_CREDITO AS TPC_DSC_TIPO_CREDITO,
        tc.DAT_EXPIRACAO_DW AS TPC_DAT_EXPIRACAO_DW,
        tc.DAT_ATUALIZACAO_DW AS TPC_DAT_ATUALIZACAO_DW,
        tc.DAT_CRIACAO_DW AS TPC_DAT_CRIACAO_DW
    FROM df_base_recarga_tecnologia f
    LEFT JOIN TIPO_CREDITO tc
        ON f.COD_TIPO_CREDITO = tc.COD_TIPO_CREDITO
""")

df_base_recarga_tipocredito.createOrReplaceTempView("df_base_recarga_tipocredito")
total_linhas = df_base_recarga_tipocredito.count()

if (total_linhas == total_recarga):
    print(log(), "Join valido Registros Base:", total_linhas)
    df_base_recarga_tipocredito.show(5)

2026-02-21 23:57:57 >>> Join valido Registros Base: 1002136510
+--------------+-----------------+-----------+----------+--------------+--------------------+--------------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+-----------------------+-----------------------+------------------+----------------------+------------------+--------------------------+--------------------------+----------------------+--------------------+--------------------+------------------------+------------------------+----------------------+----------------------+------------------+-------------------+-------------------+-------------------+------------------------+------------------------+------------------+--------------------+-----------------

In [37]:
#DIM TIPO_INSERCAO
df_TIPO_INSERCAO.createOrReplaceTempView("TIPO_INSERCAO")
df_base_recarga_tipoinsercao = spark.sql("""
       SELECT
        f.*,
        -- colunas da dimensão TIPO_INSERCAO com prefixo TPI_
        ti.DW_TIPO_INSERCAO  AS TPI_DW_TIPO_INSERCAO,
        ti.DSC_TIPO_INSERCAO AS TPI_DSC_TIPO_INSERCAO,
        ti.DAT_EXPIRACAO_DW  AS TPI_DAT_EXPIRACAO_DW,
        ti.DAT_CRIACAO_DW    AS TPI_DAT_CRIACAO_DW
    FROM df_base_recarga_tipocredito f
    LEFT JOIN TIPO_INSERCAO ti
        ON f.DW_TIPO_INSERCAO = ti.DW_TIPO_INSERCAO
""")

df_base_recarga_tipoinsercao.createOrReplaceTempView("df_base_recarga_tipoinsercao")
total_linhas = df_base_recarga_tipoinsercao.count()

if (total_linhas == total_recarga):
    print(log(), "Join valido Registros Base:", total_linhas)
    df_base_recarga_tipoinsercao.show(5)

2026-02-22 00:18:33 >>> Join valido Registros Base: 1002136510
+--------------+-----------------+-----------+----------+--------------+--------------------+--------------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+-----------------------+-----------------------+------------------+----------------------+------------------+--------------------------+--------------------------+----------------------+--------------------+--------------------+------------------------+------------------------+----------------------+----------------------+------------------+-------------------+-------------------+-------------------+------------------------+------------------------+------------------+--------------------+-----------------

In [38]:
#DIM TIPO_RECARGA
df_TIPO_RECARGA.createOrReplaceTempView("TIPO_RECARGA")

df_bases_recarga_tiporecarga = spark.sql("""
       SELECT
        f.*,
        -- colunas da dimensão TIPO_RECARGA com prefixo TR_
        tr.DW_TIPO_RECARGA  AS TR_DW_TIPO_RECARGA,
        tr.DSC_TIPO_RECARGA AS TR_DSC_TIPO_RECARGA,
        tr.DAT_EXPIRACAO_DW AS TR_DAT_EXPIRACAO_DW,
        tr.DAT_CRIACAO_DW   AS TR_DAT_CRIACAO_DW
    FROM df_base_recarga_tipoinsercao f
    LEFT JOIN TIPO_RECARGA tr
        ON f.DW_TIPO_RECARGA = tr.DW_TIPO_RECARGA
""")

df_bases_recarga_tiporecarga.createOrReplaceTempView("df_bases_recarga_tiporecarga")
total_linhas = df_bases_recarga_tiporecarga.count()

if (total_linhas == total_recarga):
    print(log(), "Join valido Registros Base:", total_linhas)
    df_bases_recarga_tiporecarga.show(5)

2026-02-22 00:39:12 >>> Join valido Registros Base: 1002136510
+--------------+-----------------+-----------+----------+--------------+--------------------+--------------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+-----------------------+-----------------------+------------------+----------------------+------------------+--------------------------+--------------------------+----------------------+--------------------+--------------------+------------------------+------------------------+----------------------+----------------------+------------------+-------------------+-------------------+-------------------+------------------------+------------------------+------------------+--------------------+-----------------

In [ ]:
# criando coluna SAFRA
df_bases_recarga = spark.sql("""
SELECT
    trunc(to_timestamp(f.DAT_INSERCAO_CREDITO, 'ddMMMyyyy:HH:mm:ss'),'MM') AS SAFRA,
    f.*
FROM df_bases_recarga_tiporecarga f
""")
df_bases_recarga.show(6)

df_bases_recarga.createOrReplaceTempView("df_bases_recarga")

+----------+--------------+-----------------+-----------+----------+--------------+--------------------+--------------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+-----------------------+-----------------------+------------------+----------------------+------------------+--------------------------+--------------------------+----------------------+--------------------+--------------------+------------------------+------------------------+----------------------+----------------------+------------------+-------------------+-------------------+-------------------+------------------------+------------------------+------------------+--------------------+------------------+--------------+-----------+------------------+----

# Salvar na camada Trusted

In [ ]:
path_trusted = os.path.join(bucket_trusted, output_trusted)
print("Trusted path:", path_trusted)

df_base_recarga.write \
    .partitionBy("SAFRA","ts_proc_partition") \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(path_trusted)

# Controle de carga

In [ ]:
df_controle = spark.sql (f"""
SELECT
    '{output_trusted}' AS name_file,
    ts_proc,
    ts_proc_partition,
    count(*) as qtd_registros
from df_bases_recarga
    GROUP BY 1,2,3
""")

df_controle.show()

+--------------------+--------------+-----------------+-------------+
|           name_file|       ts_proc|ts_proc_partition|qtd_registros|
+--------------------+--------------+-----------------+-------------+
|trusted_base_recarga|20260210105613|   20260210105613|   1002136510|
+--------------------+--------------+-----------------+-------------+



# Controle de processamento

In [ ]:
path_control = os.path.join(bucket_control, f'tb_controle_processamento_{bucket_base}_trusted')
print("Control path:", path_control)

df_controle.write \
  .mode('append') \
  .option('compression', 'snappy') \
  .parquet(path_control)